# Scraper & clean player stats in a dataframe

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException
import time
general_df = pd.read_excel(r'C:\Users\Coco\OneDrive - Erasmus University Rotterdam\Coding\Betting hltv\player_stats\general_df_test.xlsx')

# Remove the following line when finished testing
player_stats_df = pd.read_csv(r'C:\Users\Coco\OneDrive - Erasmus University Rotterdam\Coding\Betting hltv\player_stats\player_stats_df.csv')

## Scraper

In [ ]:
def open_stats_links(general_df, n):
    stats_list = []

    # Iterate through all the rows in general_df
    for index, row in general_df.iterrows():
        # Set up the WebDriver
        driver = webdriver.Firefox()

        # Get the match link of the current row
        match_link = row['Match Link']

        # Navigate to the match page
        driver.get(match_link)

        # Locate all the STATS links
        elements = WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, 'div.results-center-stats > a.results-stats')))

        # Keep track of all the open tabs
        tabs = [driver.current_window_handle]

        # Keep track of the current map number
        map_num = 1

        # Iterate through the STATS links and click on them
        for element in elements:
            if element.text == "STATS":
                stats_link = element.get_attribute("href")
                print(f"Opening STATS link: {stats_link}")

                # Open the STATS link in a new tab
                driver.execute_script("window.open('');")

                # Switch to the new tab
                driver.switch_to.window(driver.window_handles[-1])

                # Navigate to the correct STATS link in the new tab
                driver.get(stats_link)

                # Scrape the player statistics for the current tab
                player_stats = scrape_player_stats(driver, map_num)

                # Add the player statistics to the stats_list
                stats_list.extend(player_stats)

                # Print the player statistics for debugging purposes
                print(f'Player statistics for map {map_num}:')
                for stat in player_stats:
                    print(stat)
                    print('-' * 50)

                # Add the new tab to the list of open tabs
                tabs.append(driver.current_window_handle)

                # Increment the map number
                map_num += 1

                # Add a delay to ensure that the page loads completely
                time.sleep(2)

                # Close the current tab
                driver.close()

                # Switch back to the original tab
                driver.switch_to.window(tabs[0])

        # Close the original tab
        driver.close()

        # Remove the closed tab from the list of open tabs
        tabs.remove(tabs[0])

        # Close the WebDriver
        driver.quit()

    return stats_list

####################" Scrape player statistics ####################

def scrape_player_stats(driver, map_num):
    # Find all the tables with the player statistics
    tables = driver.find_elements(By.CSS_SELECTOR, 'div.stats-content > table.stats-table.totalstats')
    player_stats = []

    for table in tables:
        # Find all the rows in the table
        rows = table.find_elements(By.CSS_SELECTOR, 'tbody > tr')

        # Iterate through each row and extract the player statistics
        
        for row in rows:
            # Find all the columns in the row
            columns = row.find_elements(By.TAG_NAME, 'td')

            # Extract the player name
            player_name = columns[0].find_element(By.CSS_SELECTOR, 'a').get_attribute('textContent')
            print(f'Found player name: {player_name}')

            # Extract the kills
            kills = columns[1].get_attribute('textContent')
            print(f'Found kills: {kills}')

            # Extract the assists
            assists = columns[2].get_attribute('textContent')
            print(f'Found assists: {assists}')

            # Extract the deaths
            deaths = columns[3].get_attribute('textContent')
            print(f'Found deaths: {deaths}')

            # Extract the K/D ratio
            kdratio = columns[4].get_attribute('textContent')
            print(f'Found K/D ratio: {kdratio}')

            # Extract the K/D difference
            kddiff = columns[5].get_attribute('data-sort')
            print(f'Found K/D difference: {kddiff}')

            # Extract the average damage per round
            adr = columns[6].get_attribute('textContent')
            print(f'Found average damage per round: {adr}')

            # Extract the first kill difference
            fkdiff = columns[7].get_attribute('textContent')
            print(f'Found first kill difference: {fkdiff}')

            # Extract the rating
            rating = columns[8].get_attribute('textContent')
            print(f'Found rating: {rating}')

            # Print a separator for clarity
            print('-' * 50)

            # Create a dictionary with the player statistics and map number
            player_dict = {
                'Player': f'{player_name}',
                'Map': map_num,
                'Kills': kills,
                'Assists': assists,
                'Deaths': deaths,
                'K/D Ratio': kdratio,
                'K/D Difference': kddiff,
                'Average Damage per Round': adr,
                'First Kill Difference': fkdiff,
                'Rating': rating
            }

            # Add the dictionary to the list of player statistics
            player_stats.append(player_dict)
            WebDriverWait(driver,5)

    return player_stats

# Call the open_stats_links function and store the result in a variable
stats_list = open_stats_links(general_df, 1)

# Initialize an empty DataFrame
player_stats_df = pd.DataFrame()

# Iterate through the list of dictionaries
for i, player_stat in enumerate(stats_list):
    # Create a new DataFrame from the current dictionary
    temp_df = pd.DataFrame([player_stat])

    # Merge the new DataFrame with the existing DataFrame
    player_stats_df = pd.concat([player_stats_df, temp_df], ignore_index=True)

# Print the DataFrame
player_stats_df

## Clean player_stats

In [ ]:
def clean_kills(kills_str):
    if "(" in kills_str and ")" in kills_str:
        kills, hs_kills = kills_str.split(" (")
        hs_kills = hs_kills.replace(")", "")
        return int(kills), int(hs_kills)
    else:
        return int(kills_str), None

# Assuming 'player_stats_df' is your DataFrame
kills, hs_kills = zip(*player_stats_df["Kills"].apply(clean_kills))
player_stats_df["Kills"] = kills
player_stats_df["hs_kills"] = hs_kills

def clean_assists(assists_str):
    if "(" in assists_str and ")" in assists_str:
        assists, flash_assists = assists_str.split(" (")
        flash_assists = flash_assists.replace(")", "")
        return int(assists), int(flash_assists)
    else:
        return int(assists_str), None

# Assuming 'player_stats_df' is your DataFrame
assists, flash_assists = zip(*player_stats_df["Assists"].apply(clean_assists))
player_stats_df["Assists"] = assists
player_stats_df["flash_assists"] = flash_assists

# Print the updated DataFrame
player_stats_df

##############################################""""""""##############################################

############ Clean the columns ############

# Convert the columns to integers
int_columns = ['K/D Difference', 'First Kill Difference', 'Deaths', 'Assists', 'Kills', 'Map']
player_stats_df[int_columns] = player_stats_df[int_columns].astype(int)

# Convert the columns to floats
float_columns = ['Average Damage per Round', 'Rating']
player_stats_df[float_columns] = player_stats_df[float_columns].astype(float)

# Remove the percentage sign and convert the column to floats
player_stats_df['K/D Ratio'] = player_stats_df['K/D Ratio'].str.replace('%', '').astype(float)
player_stats_df=player_stats_df.rename(columns={"K/D Ratio": "K/D Ratio (%)"})
player_stats_df

############ Make a dictionary out of the statistics ############

# Initialize an empty dictionary to store the player statistics for each map
player_stats_dict = {}

# Iterate over the rows of the player_stats_df DataFrame
for index, row in player_stats_df.iterrows():
    # Get the map number for the current row
    map_num = f"Map {row['Map']}"
    
    # Check if the map number is already a key in the player_stats_dict
    if map_num not in player_stats_dict:
        # If the map number is not a key, initialize an empty dictionary for that map
        player_stats_dict[map_num] = {}
    
    # Create a nested dictionary for the player statistics
    player_stats = {
        'Kills': row['Kills'],
        'Assists': row['Assists'],
        'Deaths': row['Deaths'],
        'K/D Ratio (%)': row['K/D Ratio (%)'],
        'K/D Difference': row['K/D Difference'],
        'Average Damage per Round': row['Average Damage per Round'],
        'First Kill Difference': row['First Kill Difference'],
        'Rating': row['Rating']
    }
    
    # Add the nested dictionary to the player_stats_dict for the corresponding map
    player_stats_dict[map_num][row['Player']] = player_stats
player_stats_dict

# Saved file for map_and_player

In [ ]:
player_stats_df.head(3)

In [ ]:
map_and_player = general_df.copy()
map_and_player.head()

In [ ]:
# Get the amount of maps that were played in a matchup so we can extract the right 
# amount of rows aka player stats
for matchup, score in map_and_player.iterrows():
    score = score["Score"]
    #print(f"The matchup is {matchup} and the score is: {score}")
    #print('-' * 50)
    score_team1, score_team2 = score.split(" - ")
    total_score = int(score_team1) + int(score_team2)
    #print(f"The total score is: {total_score}")
#############################################################################################
for index, row in player_stats_df.iterrows():
    
    """
    print(f"The row is: {row} and it's type is: {type(row)}")
    print('-' * 50)
    print(f"and the index is: {index}")
    print('-' * 50)
    """
    # Take the data of all players that played in the same map (all 10 players aka 10 rows)
    if index % 10 == 0:
        print(f"The name of the player is :{row["Player"]}")
        dict_player = {}
        dict_player[row["Player"]] = {
            "Map" : row["Map"],
            "Kills" : row["Kills"],
            "Assists" : row["Assists"],
            "Deaths" : row["Deaths"],
            "K/D Ratio" : row["K/D Ratio (%)"],
            "K/D Difference" : row["K/D Difference"],
            "Average Damage per Round" : row["Average Damage per Round"],
            "First Kill Difference" : row["First Kill Difference"],
            "Rating" : row["Rating"],
            "hs_kills" : row["hs_kills"],
            "flash_assists" : row["flash_assists"]
        }

    # Delimit a new map being played    
    if index % 10 == 1:
        print(f"One map has been played, the index is: {index} and the map is: {row['Map']}")
        print('-' * 50)

    

In [ ]:
# CREATE THE COLUMN NAMES FOR THE MAPS AND PLAYERS

# Iterate through each row in the DataFrame
for index, row in map_and_player.iterrows():
    score = row["Score"]
    score_team1, score_team2 = score.split(" - ")
    total_score = int(score_team1) + int(score_team2)
    
    # Add columns for each map
    for map_number in range(1, total_score + 1):
        for player_number in range(1, 11):
            column_name = f"Player {player_number} map {map_number}"
            # Filter player stats for the current matchup and map using the index
            player_stat = player_stats_df[(player_stats_df.index == index) & (player_stats_df['Map'] == map_number) & (player_stats_df['Player'] == f'Player {player_number}')]
            if not player_stat.empty:
                map_and_player.loc[index, column_name] = player_stat['Stats'].values[0]
            else:
                map_and_player.loc[index, column_name] = None  # or any default value

map_and_player.head()

In [ ]:
# Iterate over the rows of the map_and_player DataFrame
for index, row in map_and_player.iterrows():
    # Determine the number of maps played in the matchup
    num_maps_played = row['Number of Maps Played']
    
    # Calculate the total number of rows to process from player_stats_df
    total_rows = num_maps_played * 10
    
    # Initialize a list to store player statistics for the current matchup
    matchup_stats = []
    
    # Iterate through the corresponding rows in player_stats_df
    for i in range(total_rows):
        player_index = index * total_rows + i
        if player_index >= len(player_stats_df):
            print(f"Warning: player_index {player_index} is out of bounds for player_stats_df with length {len(player_stats_df)}")
            continue
        
        player_row = player_stats_df.iloc[player_index]
        player_name = player_row['Player']
        
        # Create a dictionary for the player
        player_dict = player_row.drop('Player').to_dict()
        
        # Add the player dictionary to the matchup statistics
        matchup_stats.append({player_name: player_dict})
    
    # Add the matchup statistics to the appropriate column in map_and_player DataFrame
    map_and_player.at[index, 'Player Stats'] = matchup_stats
############################################################################################################
# Function to calculate the number of maps played
def calculate_maps_played(score_str):
    score_team1, score_team2 = map(int, score_str.split(' - '))
    return score_team1 + score_team2

# Apply the function to the "Score" column
map_and_player['Number of Maps Played'] = map_and_player['Score'].apply(calculate_maps_played)

# Determine the maximum number of maps played
max_maps_played = map_and_player['Number of Maps Played'].max()

# Create columns for each player's stats in each map
for map_num in range(1, max_maps_played + 1):
    for player_num in range(1, 11):
        map_and_player[f'Player {player_num} map {map_num}'] = None

# Initialize an empty dictionary to store the player statistics for each map
player_stats_dict = {}

# Iterate over the rows of the map_and_player DataFrame
for index, row in map_and_player.iterrows():
    # Determine the number of maps played in the matchup
    num_maps_played = row['Number of Maps Played']
    
    # Calculate the total number of rows to process from player_stats_df
    total_rows = num_maps_played * 10
    
    # Initialize a list to store player statistics for the current matchup
    matchup_stats = []
    
    # Iterate through the corresponding rows in player_stats_df
    for i in range(total_rows):
        player_row = player_stats_df.iloc[index * total_rows + i]
        player_name = player_row['Player']
        
        # Create a dictionary for the player
        player_dict = player_row.drop('Player').to_dict()
        
        # Add the player dictionary to the matchup statistics
        matchup_stats.append({player_name: player_dict})
    
    # Add the matchup statistics to the appropriate column in map_and_player DataFrame
    map_and_player.at[index, 'Player Stats'] = matchup_stats

# Fill the player columns based on the "Player Stats" dictionary
for index, row in map_and_player.iterrows():
    player_stats = row['Player Stats']
    for map_num in range(1, max_maps_played + 1):
        for player_num in range(1, 11):
            player_index = (map_num - 1) * 10 + (player_num - 1)
            if player_index < len(player_stats):
                player_name, player_dict = list(player_stats[player_index].items())[0]
                map_and_player.at[index, f'Player {player_num} map {map_num}'] = player_dict

# Display the updated map_and_player DataFrame
map_and_player

In [ ]:
map_and_player.head()

In [ ]:
for index, row in map_and_player.iterrows():
    for player_stats in row['Player Stats']:
        for player, stats in player_stats.items():
            print(f"Player: {player}")
            print(f"Map: {stats['Map']}")
            print(f"Kills: {stats['Kills']}")
            print(f"Assists: {stats['Assists']}")
            print(f"Deaths: {stats['Deaths']}")
            print(f"K/D Ratio (%): {stats['K/D Ratio (%)']}")
            print(f"K/D Difference: {stats['K/D Difference']}")
            print(f"Average Damage per Round: {stats['Average Damage per Round']}")
            print(f"First Kill Difference: {stats['First Kill Difference']}")
            print(f"Rating: {stats['Rating']}")
            print(f"hs_kills: {stats['hs_kills']}")
            print(f"flash_assists: {stats['flash_assists']}")
            print('-' * 50)

# Now we can extract the player stats for each map and player from the "Player Stats" column
# and create new columns in the map_and_player DataFrame to store this information and store the 
# statistics of each player per map in a column of the map_and_player DataFrame. This information is a dictionary
# with the player name as the key and the player statistics as the value. We can then extract the player statistics easily



## Testing the following 3 things
- All that needs to be done is to take the first dictionary in the list in Player Stats and place it in the column "Player 1 map 1" then the second dictionary in the list in Player Stats column place it in the column "Player 2 map 1" and etc.

In [ ]:
map_and_player.head(3)

In [ ]:
for index, row in map_and_player.iterrows():
    for player_stats in row['Player Stats']:
        for player, stats in player_stats.items():
            
            """
            print(f"Player: {player}")
            print(f"Map: {stats['Map']}")
            print(f"Kills: {stats['Kills']}")
            print(f"Assists: {stats['Assists']}")
            print(f"Deaths: {stats['Deaths']}")
            print(f"K/D Ratio (%): {stats['K/D Ratio (%)']}")
            print(f"K/D Difference: {stats['K/D Difference']}")
            print(f"Average Damage per Round: {stats['Average Damage per Round']}")
            print(f"First Kill Difference: {stats['First Kill Difference']}")
            print(f"Rating: {stats['Rating']}")
            print(f"hs_kills: {stats['hs_kills']}")
            print(f"flash_assists: {stats['flash_assists']}")
            print('-' * 50)
            """

In [ ]:
# Initialize variables to keep track of the current map number and row index
current_map = 1
row_index = 0

# Iterate over each row in the DataFrame
for index, row in map_and_player.iterrows():
    # Iterate over each dictionary in the "Player Stats" list
    for i, player_stats in enumerate(row['Player Stats']):
        # Get the player number and map number from the dictionary
        player_num = (i % 10) + 1
        map_num = list(player_stats.values())[0]['Map']

        # If the map number is different from the current map number, increment the row index
        if map_num != current_map:
            row_index += 1
            current_map = map_num

        # Store the player statistics in the corresponding column
        map_and_player.at[row_index, f'Player {player_num} map {map_num}'] = list(player_stats.values())[0]



In [ ]:
# Initialize variables to keep track of the current row index
row_index = 0

# Iterate over each row in the DataFrame
for index, row in map_and_player.iterrows():
    # Check if the value in the "Player Stats" column is a float
    if isinstance(row['Player Stats'], float):
        continue

    # Iterate over each dictionary in the "Player Stats" list
    for i, player_stats in enumerate(row['Player Stats']):
        # Get the player number and map number from the dictionary
        player_num = i + 1
        map_num = list(player_stats.values())[0]['Map']

        # Store the player statistics in the corresponding column
        map_and_player.at[row_index, f'Player {player_num} map {map_num}'] = list(player_stats.values())[0]

    # Increment the row index
    row_index += 1


In [ ]:

# Iterate over each row in the DataFrame
for index, row in map_and_player.iterrows():
    # Check if the value in the "Player Stats" column is a float
    if isinstance(row['Player Stats'], float):
        continue

    # Iterate over each dictionary in the "Player Stats" list
    for i, player_stats in enumerate(row['Player Stats']):
        # Get the player number and map number from the dictionary
        player_num = (i % 10) + 1
        map_num = list(player_stats.values())[0]['Map']

        # Store the player statistics in the corresponding column
        map_and_player.at[index, f'Player {player_num} map {map_num}'] = list(player_stats.values())[0]


In [ ]:
map_and_player.head()